# Outage Duration Uncertainty — End-to-End Show & Tell
**Plant Millbrook · Refuelling Outage OH-005**

This notebook walks the full pipeline from raw schedule data to probabilistic
outage finish-time risk:

```
Historical activities  ──►  NLP text cleaning  ──►  Semantic analog retrieval
                                                             │
                                                 Duration distribution fitting
                                                             │
                                          Schedule network (RCPSP / CPM)
                                                             │
                                      Monte Carlo uncertainty propagation
                                                             │
                                  Finish-time risk metrics (p50/p80/p90)
```

**External artefacts:**
- `demo_data.py`  — historical DB, planned activities, schedule topology
- `demo_plots.py` — reusable visualisation functions

> **Production path note:** replace the manual `OutageRecord` assembly here
> with `ActivityService.build_outage_record(p6_dataset, outage)` to ingest
> directly from a Primavera P6 `OutageDataset`.

In [ ]:
import sys, csv, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_HERE = Path(".").resolve()
_ROOT = _HERE.parent
_OUTAGE_ROOT = _ROOT.parent
for p in [str(_HERE), str(_OUTAGE_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
print("Environment ready.")

## Section 0 · Data Setup

In [ ]:
from demo_data import (
    OUTAGE_METADATA, HISTORICAL_ROWS, PLANNED_ACTIVITIES,
    SCHEDULE_TOPOLOGY, BENCHMARK_CSV,
)

print(f"Outage      : {OUTAGE_METADATA['outage_id']}  —  {OUTAGE_METADATA['outage_name']}")
print(f"Historical  : {len(HISTORICAL_ROWS)} activities "
      f"({len({r['outage_id'] for r in HISTORICAL_ROWS})} prior outages)")
print(f"Planned     : {len(PLANNED_ACTIVITIES)}")
print(f"Network nodes: {len(SCHEDULE_TOPOLOGY)}")

In [ ]:
from collections import Counter
import numpy as np

outages  = Counter(r["outage_id"]   for r in HISTORICAL_ROWS)
disc     = Counter(r["discipline"]  for r in HISTORICAL_ROWS)
overruns = sum(1 for r in HISTORICAL_ROWS
               if (r.get("actual_duration_hours") or 0) > (r.get("planned_duration_hours") or 0))
avg_slip = np.mean([
    r["actual_duration_hours"] / r["planned_duration_hours"]
    for r in HISTORICAL_ROWS
    if r.get("actual_duration_hours") and r.get("planned_duration_hours")
])
print(f"  By outage    : {dict(outages)}")
print(f"  Disciplines  : {dict(disc)}")
print(f"  Overruns     : {overruns}/{len(HISTORICAL_ROWS)}  "
      f"avg actual/planned={avg_slip:.2f}x")

## Section 1 · Text Pre-processing

P6 descriptions contain abbreviations (`MOV`, `RHR`), typos, mixed case, and
telegraphic style. The pipeline applies:
1. **AbbreviationResolver** — expands nuclear-domain abbreviations
2. **DomainSpellChecker** — corrects transcription / OCR errors

In [ ]:
with open(BENCHMARK_CSV, newline="") as f:
    benchmark_rows = list(csv.DictReader(f))
print(f"Benchmark: {len(benchmark_rows)} rows, "
      f"categories: {sorted({r['category'] for r in benchmark_rows})}")

### 1a · Step-by-step trace

In [ ]:
from outage_uncertainty.preprocessing.abbreviations import AbbreviationResolver
from outage_uncertainty.preprocessing.spell_checker import DomainSpellChecker

resolver = AbbreviationResolver()
checker  = DomainSpellChecker()

example = benchmark_rows[1]
print(f"Reference  : {example['clean_description']}")
print(f"Raw input  : {example['contaminated_description']}")
step1 = resolver.transform(example["contaminated_description"])
print(f"After abbr : {step1}")
step2 = checker.transform(step1)
print(f"After spell: {step2}")
print(f"Profile    : {example['contamination_profile']}")

### 1b · Batch spot-check (10 random descriptions)

In [ ]:
import random
random.seed(42)
sample = random.sample(benchmark_rows, 10)
print(f"{'ID':<8} {'Raw input (truncated)':<42} {'Cleaned'}")
print("-" * 95)
for row in sample:
    raw     = row["contaminated_description"]
    cleaned = checker.transform(resolver.transform(raw))
    print(f"{row['benchmark_id']:<8} {raw[:42]:<42} {cleaned[:45]}")

## Section 2 · Pre-processing Quality Benchmark

**NED**: normalised edit distance (0 = identical, 1 = completely different).  
**Exact-match rate**: fraction of descriptions fully restored by cleaning.

In [ ]:
from outage_uncertainty.visualization.plots import plot_preprocessing_benchmark
fig, _ = plot_preprocessing_benchmark(benchmark_rows)
plt.show()

## Section 3 · Completion Time Variance (NLP — Stage D)

For each planned activity the estimation service:
1. Cleans the description (Section 1 pipeline)
2. Embeds it and finds the k most semantically similar historical activities
3. Fits a `DurationDistribution` from their actual completion times
4. Assigns a confidence tier: `data_supported` / `sme_informed` / `low_confidence`

In [ ]:
from outage_uncertainty.api.facade import build_duration_uncertainty_service
service = build_duration_uncertainty_service()
print("Estimation service ready.")

In [ ]:
results = []
for act in PLANNED_ACTIVITIES:
    est = service.estimate_activity(
        query_row=act,
        historical_rows=HISTORICAL_ROWS,
    )
    results.append(est)
    dist = est.estimated_distribution
    p50 = f"{dist.p50:.1f} h" if dist else "—"
    print(f"  {act['activity_id']:<18}  tier={est.confidence_tier:<22} "
          f"support={est.support_count:<4}  p50={p50}")

### 3a · Planned vs P50 estimate summary

In [ ]:
from outage_uncertainty.visualization.plots import plot_duration_summary_bars
fig, ax = plot_duration_summary_bars(PLANNED_ACTIVITIES, results)
plt.show()

### 3b · Analogue deep-dive

`Q-MCP-2B` — data-supported (5+ analogues from identical component family).  
`Q-DCS-UPGRDE` — low confidence / epistemic (no close historical match).

In [ ]:
_SEP = "─" * 64
est_by_id = {act["activity_id"]: est
             for act, est in zip(PLANNED_ACTIVITIES, results)}

for target_id in ("Q-MCP-2B", "Q-DCS-UPGRDE"):
    act = next(a for a in PLANNED_ACTIVITIES if a["activity_id"] == target_id)
    est = est_by_id[target_id]
    print(f"\n{_SEP}")
    print(f"Activity  : {target_id}")
    print(f"Query     : {act['raw_description']}")
    print(f"Tier      : {est.confidence_tier}  |  "
          f"support={est.support_count}  |  type={est.uncertainty_type}")
    if est.estimated_distribution:
        d = est.estimated_distribution
        p80 = getattr(d, "p80", None)
        p80_str = f"  P80={p80:.1f} h" if p80 else ""
        print(f"P50={d.p50:.1f} h{p80_str}")
    print("Top analogues:")
    for i, analog in enumerate(est.matched_cases[:3], 1):
        desc  = getattr(analog, "candidate_activity_id", str(analog))[:54]
        dur   = getattr(analog, "candidate_duration_hours", "?")
        score = getattr(analog, "total_score", "?")
        score_str = f"  score={score:.3f}" if isinstance(score, float) else ""
        print(f"  {i}. {desc:<55} dur={dur}h{score_str}")
print(f"\n{_SEP}")

### 3c · Duration distributions (routine vs extended)

In [ ]:
from outage_uncertainty.visualization.plots import plot_duration_distributions, plot_analog_scatter

fig, _ = plot_duration_distributions(PLANNED_ACTIVITIES, results)
plt.show()

In [ ]:
fig, ax = plot_analog_scatter(PLANNED_ACTIVITIES, results)
plt.show()

### 3d · Routine vs disruption-driven execution modes

**How step 3 of the duration estimation works (Slide 6):**

Historical analog durations are rarely drawn from a single population.
Two distinct regimes appear in outage data:

| Mode | Cause | Typical signature |
|---|---|---|
| **Routine execution** | Work proceeds as scoped — no surprises | Tightly clustered around the planned duration |
| **Disruption-driven** | Scope expansion, rework, parts delay, access conflict | Outliers 1.5–3× above the routine cluster |

The `OutlierHandler` uses an **IQR upper fence** (Q3 + 1.5 × IQR) to separate the two populations.
`DistributionFitter.fit_from_separation()` then produces a **mixture distribution**:

- `samples` → routine pool; drives P50/P80 for *clean-run* planning
- `extended_samples` → disruption pool; weight = fraction of historical jobs that ran in disrupted mode
- `mixture_weight` → probability a future job will enter disrupted mode
- `parameters["mixture_p80"]` → the *true* 80th-percentile anchor once disruption probability is included

The gap **mixture P80 − routine P80** is the contingency that is *silently missing* when you fit a single lognormal to the pooled data.

In [ ]:
from outage_uncertainty.uncertainty.outlier_handler import OutlierHandler
from outage_uncertainty.uncertainty.distribution_fitter import DistributionFitter
from outage_uncertainty.visualization.plots import plot_routine_vs_disruption

# ── Demonstrate with MCP-2B (pump seal replacement) ──────────────────────
# Illustrative analog pool mixing routine and disruption-driven cases.
# Routine cluster  : typical MCP seal jobs (~20-27 h)
# Disruption cases : scope-expanded jobs (42-62 h) — e.g. secondary seal
#   damage found on disassembly, triggering parts wait + rework.
pump_seal_analogs = [
    22.0, 23.5, 21.5, 24.0, 26.0, 20.5, 25.5, 22.5,  # routine
    23.0, 21.0, 24.5, 26.5, 20.0, 27.0,               # routine (cont.)
    45.0, 53.5, 61.0,                                   # disruption
]

handler = OutlierHandler(strategy="iqr")
fitter  = DistributionFitter()

sep  = handler.separate(pump_seal_analogs)
dist = fitter.fit_from_separation(sep)

n_total    = sep.n_total
n_routine  = sep.n_routine
n_extended = len(sep.extended)
ext_frac   = sep.extended_fraction

print("── Separation result ──────────────────────────────────────")
print(f"  Strategy      : {sep.method}")
print(f"  IQR fence     : {sep.threshold:.1f} h")
print(f"  Routine jobs  : {n_routine}  ({(1-ext_frac)*100:.0f}% of sample)")
print(f"  Disrupted jobs: {n_extended}  ({ext_frac*100:.0f}% of sample)")
print()
print("── Distribution percentiles ───────────────────────────────")
print(f"  Routine P50   : {dist.p50:.1f} h  (median — clean-run planning baseline)")
print(f"  Routine P80   : {dist.p80:.1f} h  (ignores disruption probability)")
mix_p80 = dist.parameters.get("mixture_p80")
mix_p90 = dist.parameters.get("mixture_p90")
if mix_p80:
    gap = mix_p80 - dist.p80
    print(f"  Mixture P80   : {mix_p80:.1f} h  (accounts for {ext_frac*100:.0f}% disruption risk)")
    print(f"  Contingency gap: +{gap:.1f} h hidden when disruption pooled away")
if mix_p90:
    print(f"  Mixture P90   : {mix_p90:.1f} h")

fig, _ = plot_routine_vs_disruption(
    dist, "Q-MCP-2B  (MCP seal replacement)",
    planned_hours=20.0,
)
import matplotlib.pyplot as plt
plt.show()


## Section 4 · Schedule Network Assembly (RCPSP / CPM)

The six planned activities are embedded in a 9-node DAG that mirrors a
typical PWR refuelling outage topology.  Each `ScheduleActivity` carries
the `DurationDistribution` fitted in Section 3.

```
Q-INIT (4h)
├── Q-MCP-2B (20h) ──► Q-REPL (6h) ──────────────────────────┐
├── Q-MSIV-2 (10h) ───────────────────────────────────────────►│
├── Q-HX-CCW (10h) ──► Q-LT-CALIB (3h) ──► Q-BKR-4A (8h) ──►│── Q-END (2h)
└── Q-DCS-UPGRDE (40h) ────────────────────────────────────────►│
```

Deterministic baseline CP: **Q-INIT → Q-DCS-UPGRDE → Q-END = 46 h**

In [ ]:
from outage_uncertainty.domain.schedule import ScheduleActivity
from outage_uncertainty.schedule_risk.schedule_graph import ScheduleNetwork

est_by_id = {act["activity_id"]: est
             for act, est in zip(PLANNED_ACTIVITIES, results)}

def _dist(aid):
    est = est_by_id.get(aid)
    return est.estimated_distribution if (est and est.estimated_distribution) else None

sched_activities = [
    ScheduleActivity(
        activity_id=aid,
        name=name,
        predecessors=preds,
        successors=succs,
        baseline_duration_hours=baseline_h,
        duration_distribution=_dist(aid),
    )
    for aid, name, preds, succs, baseline_h in SCHEDULE_TOPOLOGY
]

network = ScheduleNetwork(sched_activities)

baseline_durations = {a.activity_id: a.baseline_duration_hours for a in sched_activities}
baseline_result    = network.compute_critical_path(baseline_durations)
baseline_cp        = baseline_result["cp_time"]

n_stochastic = sum(1 for a in sched_activities if a.duration_distribution is not None)
print(f"Network nodes        : {len(sched_activities)}")
print(f"Baseline CP time     : {baseline_cp:.1f} h")
print(f"Baseline CP path     : {' -> '.join(baseline_result['cp_path'])}")
print(f"Stochastic activities: {n_stochastic}  (with DurationDistribution from Stage D)")
print(f"Deterministic nodes  : {len(sched_activities) - n_stochastic}")

## Section 5 · Monte Carlo Uncertainty Quantification

`MonteCarloSimulator` repeats CPM 2 000 times, drawing one sample from each
activity's `DurationDistribution` per iteration. `CriticalPathRiskAnalyzer`
aggregates the resulting CP-time distribution into risk metrics.

| Metric | Meaning |
|--------|---------|
| `p50_finish` | Median outage finish |
| `p80_finish` | Planning-buffer target (80th pct) |
| `robustness` | Prob of finishing ≤ baseline |
| `criticality_index` | Fraction of runs on CP |
| `expected_drag` | Avg hours added when on CP |
| `cp_sensitivity` | Correlation with overrun (ρ) |

In [ ]:
from outage_uncertainty.schedule_risk.monte_carlo import MonteCarloSimulator
from outage_uncertainty.schedule_risk.scenario_runner import ScenarioRunner

N_SAMPLES = 2_000

runner = ScenarioRunner()
risk   = runner.run(network, baseline_cp_time=baseline_cp, n_samples=N_SAMPLES)

print(f"Simulations      : {N_SAMPLES:,}")
print(f"Baseline CP      : {baseline_cp:.1f} h")
print(f"P50 finish       : {risk['p50_finish']:.1f} h  "
      f"(+{risk['p50_finish'] - baseline_cp:.1f} h)")
print(f"P80 finish       : {risk['p80_finish']:.1f} h  "
      f"(+{risk['p80_finish'] - baseline_cp:.1f} h)")
print(f"P90 finish       : {risk['p90_finish']:.1f} h")
print(f"Robustness       : {risk['robustness']*100:.1f}%  "
      f"(overrun risk: {(1-risk['robustness'])*100:.1f}%)")

In [ ]:
# Raw cp_times for plotting
simulator  = MonteCarloSimulator(network, n_samples=N_SAMPLES)
sim_result = simulator.run()
cp_times   = sim_result.cp_times

### 5a · Outage finish-time distribution

In [ ]:
from outage_uncertainty.visualization.plots import plot_finish_distribution
fig, ax = plot_finish_distribution(cp_times, risk, baseline_cp)
plt.show()

### 5b · Activity risk ranking

Activities sorted by CP sensitivity (schedule leverage score).

In [ ]:
from outage_uncertainty.visualization.plots import plot_activity_risk_ranking
work_ids = [a["activity_id"] for a in PLANNED_ACTIVITIES]
fig, _ = plot_activity_risk_ranking(risk, work_ids)
plt.show()

### 5c · Critical-path route frequency heatmap

In [ ]:
from outage_uncertainty.visualization.plots import plot_cp_path_heatmap
fig, ax = plot_cp_path_heatmap(sim_result.cp_paths, top_n=6)
plt.show()

## Section 6 · Summary

| Stage | Component | Key output |
|-------|-----------|------------|
| Pre-processing | `AbbreviationResolver` + `DomainSpellChecker` | Canonical descriptions |
| NLP retrieval | `EmbeddingSemanticScorer` + `ContextSimilarityScorer` | k-NN historical analogues |
| Distribution fitting | `DurationDistribution` | p50 / p80 / p90 per activity |
| Schedule assembly | `ScheduleNetwork` (CPM, FS-lag) | Baseline CP = 46 h |
| UQ propagation | `MonteCarloSimulator` × 2 000 | CP-time distribution |
| Risk metrics | `CriticalPathRiskAnalyzer` | Robustness, criticality, drag, sensitivity |

---

**Production path (Primavera P6 → risk metrics):**
```python
from outage_uncertainty.services.activity_service import ActivityService

outage_record = ActivityService(ingestion_workflow).build_outage_record(
    p6_dataset, outage          # converts P6 XER → OutageRecord in one call
)
result = OutageRiskWorkflow(
    estimator_workflow = estimator,
    schedule_builder   = OutageRecordScheduleBuilder(),
    scenario_runner    = ScenarioRunner(analyzer=CriticalPathRiskAnalyzer()),
).run(outage_record, historical_activities=[...])
```

In [ ]:
print(f"OH-005 Risk Summary")
print("=" * 56)
print(f"  Baseline CP              : {baseline_cp:.0f} h")
print(f"  P50 finish               : {risk['p50_finish']:.1f} h  "
      f"(+{risk['p50_finish'] - baseline_cp:.1f} h)")
print(f"  P80 finish               : {risk['p80_finish']:.1f} h  "
      f"(+{risk['p80_finish'] - baseline_cp:.1f} h)")
print(f"  P90 finish               : {risk['p90_finish']:.1f} h")
print(f"  Overrun risk             : {(1-risk['robustness'])*100:.1f}%")
print()
print("  Activity risk ranking (by CP sensitivity ρ):")
ci  = risk["criticality_index"]
drg = risk["expected_drag"]
sen = risk["cp_sensitivity"]
work_ids = [a["activity_id"] for a in PLANNED_ACTIVITIES]
ranked   = sorted(work_ids, key=lambda k: sen.get(k, 0), reverse=True)
for aid in ranked:
    print(f"    {aid:<18}  CI={ci.get(aid,0)*100:4.0f}%  "
          f"drag={drg.get(aid,0):4.1f} h  rho={sen.get(aid,0):.3f}")